g# Reliable and Adaptive Agentic RAG System (Step 3)

This notebook extends the Step 2 multi-agent retrieval system with reliability, adaptation, recovery, and trust mechanisms.

The goal is to make the RAG system more reliable when evidence is:

weak
incomplete
ambiguous
contradictory

The notebook implements multiple reliability mechanisms and adaptive orchestration behaviors.

## 1. Installation

In [2]:
!pip install -q pandas numpy pytrec_eval transformers accelerate

  error: subprocess-exited-with-error
  
  × Building wheel for pytrec_eval (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [41 lines of output]
      Fetching trec_eval from https://github.com/usnistgov/trec_eval/archive/v9.0.8.tar.gz.
      C:\Users\stude\AppData\Local\Temp\pip-build-env-sp1fvrnj\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: pytrec_eval).
      
              Available configuration options are listed in:
              https://setuptools.pypa.io/en/latest/userguide/declarative_config.

## 2. Imports

In [3]:
import re
import time
import random
import numpy as np
import pandas as pd

from collections import defaultdict

## 3. Load Step 2 Components

This section imports the orchestration strategies and retrievers
from the Step 2 notebook implementation.

In [4]:
%run multi-agent-step-2_strategy-A.ipynb

# --- Wrapper functions for Step 3 compatibility ---
# The legacy notebook defines orchestrator CLASSES with .run() methods.
# Step 3 expects FUNCTIONS that return (docs, trace).
# These wrappers bridge the two interfaces.

def confidence_orchestrate(query, top_k=5):
    """Run ConfidenceOrchestrator and return (answer, docs, trace)."""
    answer, docs, trace = orchestrator.run(query, top_k=top_k)
    return answer, docs, trace

def waterfall_orchestrate(query, top_k=5):
    """Run WaterfallOrchestrator and return (answer, docs, trace)."""
    answer, docs, trace = waterfall_orchestrator.run(query, top_k=top_k)
    return answer, docs, trace

def voting_orchestrate(query, top_k=5):
    """Run VotingOrchestrator (equal weights) and return (answer, docs, trace)."""
    answer, docs, trace = voting_orchestrator.run(query, top_k=top_k)
    return answer, docs, trace

print('Step 2 orchestrators loaded. Wrappers ready.')


## 4. Utility Functions

In [5]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return set(text.split())

def overlap_score(a, b):

    ta = normalize(a)
    tb = normalize(b)

    if len(ta) == 0:
        return 0.0

    return len(ta & tb) / len(ta)

## 5. Mechanisms

### 5.1 Evidence Sufficiency Estimation (A)
This module estimates whether the retrieved evidence is sufficient.

Signals used:

- overlap between query and retrieved chunks
- number of supporting chunks
- retrieval fusion score

In [6]:
class EvidenceSufficiencyAgent:
    """
    Checks: Do we have enough evidence to answer this question?
    
    How it works:
    - Count how many query words appear in each retrieved document.
    - Calculate average overlap across top 5 docs.
    - If average >= 0.15, evidence is considered sufficient.
    
    Why it matters:
    Prevents the system from answering when retrieval returns
    irrelevant or off-topic documents.
    """

    def assess(self, query, docs):

        if len(docs) == 0:
            return {
                "sufficient": False,
                "score": 0.0,
            }

        q_tokens = normalize(query)

        support_scores = []

        for d in docs[:5]:

            txt = d.page_content
            d_tokens = normalize(txt)

            overlap = len(q_tokens & d_tokens)
            overlap = overlap / max(len(q_tokens), 1)

            support_scores.append(overlap)

        avg_support = np.mean(support_scores)

        sufficient = avg_support >= 0.15

        return {
            "sufficient": sufficient,
            "score": round(float(avg_support), 3),
            "supporting_chunks": len([
                x for x in support_scores if x > 0.1
            ])
        }


### %.2 Groundness / Support Verification (B)
The answer is verified against retrieved evidence.

In [7]:
class GroundednessAgent:
    """
    Checks: Is the answer actually supported by the retrieved documents?
    
    How it works (dual-threshold):
    1. Global check: answer words must overlap >= 45% with ALL docs combined.
       (ensures the answer is broadly supported across evidence)
    2. Per-doc check: at least ONE doc must share >= 25% of answer words.
       (ensures the answer is strongly supported by at least one source)
    Both thresholds must pass. Stopwords (the, is, of...) are ignored.
    
    Why it matters:
    Prevents hallucination — answers that sound plausible but are not
    actually found in the source documents.
    """

    # Lightweight stopword list for cleaner token overlap
    _STOP = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'shall', 'should', 'can', 'could', 'may', 'might', 'must',
        'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from', 'as', 'to',
        'and', 'or', 'but', 'if', 'then', 'than', 'so', 'yet',
        'it', 'its', 'this', 'that', 'these', 'those',
        'i', 'you', 'he', 'she', 'we', 'they', 'me', 'him', 'her',
        'us', 'them', 'my', 'your', 'his', 'our', 'their',
        'what', 'which', 'who', 'when', 'where', 'why', 'how',
    }

    def _tokens(self, text):
        """Return set of non-stopword tokens."""
        return normalize(text) - self._STOP

    def verify(self, answer, docs,
               min_global_overlap=0.45,
               min_per_doc_overlap=0.25):

        ans_tokens = self._tokens(answer)

        if len(ans_tokens) == 0:
            return False

        # --- global overlap: answer vs union of all docs ---
        all_doc_tokens = set()
        per_doc_overlaps = []

        for d in docs[:5]:
            txt = d.page_content or ''
            doc_tokens = self._tokens(txt)
            all_doc_tokens |= doc_tokens
            per_doc = len(ans_tokens & doc_tokens) / max(len(ans_tokens), 1)
            per_doc_overlaps.append(per_doc)

        global_overlap = len(ans_tokens & all_doc_tokens) / max(len(ans_tokens), 1)
        best_doc_overlap = max(per_doc_overlaps) if per_doc_overlaps else 0.0

        grounded = (
            global_overlap >= min_global_overlap and
            best_doc_overlap >= min_per_doc_overlap
        )

        return grounded



### 5.3 Contradiction Detection (C)
This mechanism checks whether retrieved chunks contain conflicting statements.

Lightweight heuristic:

- detect conflicting keywords
- detect opposite numeric claims

In [8]:
class ContradictionAgent:
    """
    Checks: Do the retrieved documents contradict each other?
    
    How it works:
    - Scans top 5 documents for opposing keyword pairs.
    - Example: if one doc says 'yes' and another says 'no'
      on the same topic, that's a contradiction.
    - Keyword pairs: (yes/no), (increase/decrease), (true/false).
    
    Why it matters:
    Catches conflicting evidence that would produce unreliable answers.
    Future upgrade: LLM-based semantic contradiction detection.
    """

    CONTRADICTIONS = [
        ("yes", "no"),
        ("increase", "decrease"),
        ("true", "false"),
    ]

    def detect(self, docs):

        texts = [d.page_content.lower() for d in docs[:5]]

        for a, b in self.CONTRADICTIONS:

            has_a = any(a in t for t in texts)
            has_b = any(b in t for t in texts)

            if has_a and has_b:
                return {
                    "contradiction": True,
                    "reason": f"Detected conflict: {a} vs {b}"
                }

        return {
            "contradiction": False,
            "reason": "No obvious contradiction detected"
        }


### 5.4 Clarification Strategy (D)
The system detects ambiguous or underspecified questions.

In [9]:
class ClarificationAgent:
    """
    Checks: Is the user's question clear enough to answer?
    
    How it works:
    - Too short? (<= 3 words) → likely underspecified.
    - Contains vague pronouns? ('it', 'they', 'this') → ambiguous reference.
    If either rule matches, ask the user to clarify.
    
    Why it matters:
    Prevents the system from guessing when the user's intent is unclear.
    Better to ask than to answer the wrong question.
    """

    def needs_clarification(self, query):

        q = query.lower().strip()

        ambiguous = [
            "it",
            "they",
            "this",
            "that",
        ]

        short_query = len(q.split()) <= 3

        ambiguous_ref = any(x in q.split() for x in ambiguous)

        if short_query or ambiguous_ref:
            return True

        return False

    def clarification_question(self, query):

        return (
            "Could you clarify your question or provide "
            "more specific details?"
        )


### 5.5 Abstention Mechanism (E)
The system abstains when evidence is weak or contradictory.

In [10]:
class AbstentionAgent:
    """
    Decides: Should the system refuse to answer?
    
    How it works:
    Abstains (says 'I cannot answer reliably') if ANY of:
    - Evidence is insufficient (low overlap with docs)
    - Answer is not grounded (not supported by evidence)
    - Documents contradict each other
    
    Why it matters:
    A system that knows when it does NOT know is more trustworthy
    than one that always guesses. This is the 'I don't know' safety net.
    """

    def should_abstain(self,
                       sufficiency,
                       grounded,
                       contradiction):

        if not sufficiency["sufficient"]:
            return True

        if contradiction["contradiction"]:
            return True

        if not grounded:
            return True

        return False


### 5.6 Self-Reflection / Critique Loop (F)
The critic reviews the draft answer.

In [11]:
import re

class CriticAgent:
    """
    Reviews the draft answer and reports any quality issues.
    
    Checks performed:
    1. Grounding: flags if answer is weakly supported by documents.
    2. Contradiction: flags if evidence conflicts.
    3. Length: flags if answer is suspiciously short (< 3 words).
    4. Temporal coherence (entity_temporal queries only):
       - Finds all years mentioned in the answer.
       - Checks if at least one year is within +/- 10 of the query year.
       - Catches answers that reference the wrong decade entirely.
    
    Why it matters:
    The Critic is the final quality gate before the answer reaches the user.
    It catches subtle errors that other agents might miss.
    """

    _YEAR_RE = re.compile(r'\b(1\d{3}|20\d{2})\b')

    def _temporal_coherent(self, answer, query_year, window=10):
        """
        Answer must mention at least one year within +/- window of query_year.
        Catches answers that are grounded but reference the wrong decade.
        """
        found = [int(m) for m in self._YEAR_RE.findall(answer)]
        return any(abs(y - query_year) <= window for y in found)

    def critique(self,
                 answer,
                 grounded,
                 contradiction,
                 query_type=None,
                 query_year=None):

        feedback = []

        if not grounded:
            feedback.append("Answer weakly supported")

        if contradiction["contradiction"]:
            feedback.append("Evidence conflict detected")

        if len(answer.split()) < 3:
            feedback.append("Answer too short")

        # Temporal coherence check for entity_temporal queries
        if query_type == 'entity_temporal' and query_year:
            temporal_ok = self._temporal_coherent(answer, query_year, window=10)
            if not temporal_ok:
                feedback.append(
                    f"Temporal mismatch: answer references wrong time period (expected around {query_year})"
                )

        if len(feedback) == 0:
            feedback.append("Answer appears acceptable")

        return feedback



### 5.7 Recovery Mechanism (G)
The system changes behavior dynamically.

Recovery actions:

- switch retrieval strategy
- rewrite query
- move to clarification mode
- abstain

In [12]:
class RecoveryAgent:
    """
    Suggests and executes recovery actions when the first attempt fails.
    
    Actions:
    - switch_strategy: try a different retrieval strategy
      (e.g., confidence → voting) to get better evidence.
    - rewrite_query: add context to the query
      (e.g., append 'ETH Zurich') to improve retrieval.
    - none: no recovery needed, everything looks good.
    
    Why it matters:
    Instead of giving up, the system tries to fix the problem.
    This is the 'adaptation' part of Reliable Adaptive RAG.
    """

    def rewrite_query(self, query):

        return query + " ETH Zurich"

    def recover(self,
                query,
                current_strategy,
                sufficiency,
                contradiction):

        if contradiction["contradiction"]:

            return {
                "action": "switch_strategy",
                "new_strategy": "voting"
            }

        if not sufficiency["sufficient"]:

            rewritten = self.rewrite_query(query)

            return {
                "action": "rewrite_query",
                "query": rewritten
            }

        return {
            "action": "none"
        }


### 5.8 Trust / Confidence Scoring (H)
The confidence score combines:

- evidence sufficiency
- groundedness
- contradiction detection

In [14]:
class TrustAgent:
    """
    Computes a single confidence score from all reliability signals.
    
    Formula:
    trust = 0.6 * sufficiency_score + 0.3 * groundedness_bonus - 0.4 * contradiction_penalty
    
    Score ranges:
    - HIGH (> 0.7): strong evidence, confident answer.
    - MEDIUM (0.4-0.7): acceptable but not perfect.
    - LOW (< 0.4): weak evidence, consider abstaining.
    
    Why it matters:
    Turns multiple checks into one easy-to-understand number.
    Users and downstream systems can act on this single score.
    """

    def score(self,
              sufficiency,
              grounded,
              contradiction):

        score = 0.0

        score += sufficiency["score"] * 0.6

        if grounded:
            score += 0.3

        if contradiction["contradiction"]:
            score -= 0.4

        score = max(0.0, min(1.0, score))

        if score > 0.7:
            label = "HIGH"
        elif score > 0.4:
            label = "MEDIUM"
        else:
            label = "LOW"

        return {
            "score": round(float(score), 3),
            "label": label
        }


## 6. Adaptive Reliable RAG Orchestrator
This orchestrator integrates all reliability mechanisms.

In [15]:
class ReliableAdaptiveRAG:
    """
    Main orchestrator: wraps the Step 2 retrieval engine with
    8 reliability agents to produce trustworthy answers.
    
    Decision flow (4 branches):
    1. CLARIFY → if query is ambiguous (short / pronouns)
    2. ABSTAIN → if evidence is weak, ungrounded, or contradictory
    3. RECOVER → if first attempt fails, retry with new strategy/query
    4. ANSWER → if all checks pass, return the synthesized answer
    
    Returns a unified trace dict matching Step 2 §2.4:
    {decision, reason, signals, intermediate, final_answer, trace_log}
    """

    def __init__(self, ablate=None):
        self.ablate = set(ablate or [])

        self.sufficiency = EvidenceSufficiencyAgent()
        self.groundedness = GroundednessAgent()
        self.contradiction = ContradictionAgent()
        self.clarification = ClarificationAgent()
        self.abstention = AbstentionAgent()
        self.critic = CriticAgent()
        self.recovery = RecoveryAgent()
        self.trust = TrustAgent()

        self._draft_answer = None
        self._last_strategy = None

    def retrieve(self, query, strategy="confidence", top_k=5):
        if strategy == "waterfall":
            answer, docs, trace = waterfall_orchestrate(query, top_k)
        elif strategy == "voting":
            answer, docs, trace = voting_orchestrate(query, top_k)
        else:
            answer, docs, trace = confidence_orchestrate(query, top_k)

        self._draft_answer = answer
        self._last_strategy = strategy
        return docs, trace

    def generate_answer(self, docs):
        if len(docs) == 0:
            return "NOT FOUND"

        # Use orchestrator's synthesized answer if available
        if self._draft_answer and len(self._draft_answer.strip()) > 5:
            return self._draft_answer

        # Fallback: truncate first doc (legacy placeholder)
        return docs[0].page_content[:250]

    _YEAR_RE = re.compile(r'\b(1\d{3}|20\d{2})\b')

    def _extract_year(self, query):
        """Simple year extractor for temporal coherence checks."""
        m = self._YEAR_RE.search(query)
        return int(m.group()) if m else None

    def _ablated(self, agent_name):
        return agent_name in self.ablate

    def _compute_signals(self, query, docs, answer, ablated, query_type=None):
        """Run all reliability agents and return signals dict."""
        trace_log = []

        suff = self.sufficiency.assess(query, docs)
        trace_log.append(f"Sufficiency: {suff['sufficient']} (score={suff['score']})")

        if not "groundedness" in ablated:
            grounded = self.groundedness.verify(answer, docs)
            trace_log.append(f"Groundedness: {grounded}")
        else:
            grounded = True
            trace_log.append("Groundedness: ABATED")

        if not "contradiction" in ablated:
            contradiction = self.contradiction.detect(docs)
            trace_log.append(f"Contradiction: {contradiction['contradiction']}")
        else:
            contradiction = {"contradiction": False}
            trace_log.append("Contradiction: ABATED")

        trust = self.trust.score(suff, grounded, contradiction)
        trace_log.append(f"Trust: {trust['score']} ({trust['label']})")

        if not "critic" in ablated:
            critique = self.critic.critique(
                answer, grounded, contradiction,
                query_type=query_type, query_year=self._extract_year(query),
            )
            trace_log.append(f"Critique: {len(critique)} issues")
        else:
            critique = []
            trace_log.append("Critique: ABATED")

        abstain = self.abstention.should_abstain(suff, grounded, contradiction)

        signals = {
            "evidence_sufficiency": round(float(suff["score"]), 3),
            "grounding_score": 1.0 if grounded else 0.0,
            "has_contradictions": contradiction["contradiction"],
            "query_ambiguous": self.clarification.needs_clarification(query),
            "trust_score": trust["score"],
        }

        return {
            "suff": suff,
            "grounded": grounded,
            "contradiction": contradiction,
            "trust": trust,
            "critique": critique,
            "abstain": abstain,
            "signals": signals,
            "trace_log": trace_log,
        }

    def run(self,
            query,
            strategy="confidence",
            ablate=None,
            query_type=None,
            query_year=None):

        ablated = set(ablate or []) | self.ablate
        trace_log = []
        retry_count = 0
        recovery_action = None
        strategy_used = strategy

        # --- Clarification branch ---
        if self.clarification.needs_clarification(query):
            trace_log.append("Clarification triggered")
            return {
                "decision": "clarify",
                "reason": "Query is ambiguous (short or contains pronouns)",
                "signals": {
                    "evidence_sufficiency": 0.0,
                    "grounding_score": 0.0,
                    "has_contradictions": False,
                    "query_ambiguous": True,
                    "trust_score": 0.0,
                },
                "intermediate": {
                    "strategy_used": strategy,
                    "recovery_action": None,
                    "retry_count": 0,
                },
                "final_answer": None,
                "trace_log": trace_log,
            }

        # --- First retrieval pass ---
        docs, retrieval_trace = self.retrieve(query, strategy)
        trace_log.extend(retrieval_trace)
        answer = self.generate_answer(docs)

        result = self._compute_signals(query, docs, answer, ablated, query_type)
        trace_log.extend(result["trace_log"])

        # --- Recovery branch: try to RESCUE before abstaining ---
        # Recovery now runs WHEN the system would abstain (low reliability),
        # giving the pipeline a chance to fix the problem before giving up.
        if result["abstain"] and not "recovery" in ablated:
            recovery = self.recovery.recover(
                query, strategy, result["suff"], result["contradiction"]
            )
            recovery_action = recovery["action"]
            trace_log.append(f"Low reliability -> recovery action: {recovery_action}")

            if recovery_action == "switch_strategy":
                new_strategy = recovery["new_strategy"]
                trace_log.append(f"RETRY: switching to {new_strategy}")
                docs, retrieval_trace = self.retrieve(query, new_strategy)
                trace_log.extend(retrieval_trace)
                answer = self.generate_answer(docs)
                result = self._compute_signals(query, docs, answer, ablated, query_type)
                trace_log.extend(result["trace_log"])
                retry_count = 1
                strategy_used = new_strategy

            elif recovery_action == "rewrite_query":
                rewritten = recovery["query"]
                trace_log.append(f"RETRY: rewritten query -> {rewritten}")
                docs, retrieval_trace = self.retrieve(rewritten, strategy)
                trace_log.extend(retrieval_trace)
                answer = self.generate_answer(docs)
                result = self._compute_signals(query, docs, answer, ablated, query_type)
                trace_log.extend(result["trace_log"])
                retry_count = 1

        # --- Abstention branch (runs after any recovery attempt) ---
        if result["abstain"]:
            if retry_count > 0:
                abstain_reason = "Recovery attempted but evidence still unreliable"
            else:
                abstain_reason = "Evidence insufficient, ungrounded, or contradictory"
            trace_log.append("System abstained")
            return {
                "decision": "abstain",
                "reason": abstain_reason,
                "signals": result["signals"],
                "intermediate": {
                    "strategy_used": strategy_used,
                    "recovery_action": recovery_action,
                    "retry_count": retry_count,
                },
                "final_answer": None,
                "trace_log": trace_log,
            }

        # --- Answer branch ---
        if retry_count > 0:
            answer_reason = "Recovery succeeded; all reliability checks passed"
        else:
            answer_reason = "All reliability checks passed"
        trace_log.append("Answer generated successfully")
        return {
            "decision": "answer",
            "reason": answer_reason,
            "signals": result["signals"],
            "intermediate": {
                "strategy_used": strategy_used,
                "recovery_action": recovery_action,
                "retry_count": retry_count,
            },
            "final_answer": answer,
            "trace_log": trace_log,
        }



## 7. Initialize System

In [16]:
rag_system = ReliableAdaptiveRAG()

## 8. Example Queries
This section demonstrates:

- clarification behavior
- abstention
- recovery
- contradiction handling
- successful answer generation

In [18]:
queries = [
    "Who received ERC grants at ETH?",
    "How does ETH support innovation?",
    "it",
    "What research areas are important at ETH Zurich?"
]

for q in queries:

    print("\n" + "=" * 80)
    print("QUERY:", q)

    result = rag_system.run(q)

    print("\nDECISION:", result["decision"])
    print("REASON:", result["reason"])
    print("ANSWER:", result["final_answer"] or "(none)")

    print("\nSIGNALS:")
    for k, v in result["signals"].items():
        print(f"  {k}: {v}")

    print("\nINTERMEDIATE:")
    for k, v in result["intermediate"].items():
        print(f"  {k}: {v}")

    print("\nTRACE:")
    for t in result["trace_log"]:
        print("-", t)



QUERY: Who received ERC grants at ETH?


NameError: name 'confidence_orchestrate' is not defined

## 9. Benchmark Evaluation
This section evaluates:

- reliability scores
- latency
- system behavior
- abstention frequency

In [19]:
results = []

# Try eval_qa_data first (loaded from strategy-A notebook),
# fallback to qa_data if available.
_qa_data = eval_qa_data if "eval_qa_data" in globals() else (qa_data if "qa_data" in globals() else [])

N = min(20, len(_qa_data))

for i in range(N):

    q = _qa_data[i]["question"]

    start = time.time()

    result = rag_system.run(q)

    runtime = time.time() - start

    trust_score = result["signals"]["trust_score"]

    results.append({
        "query": q,
        "decision": result["decision"],
        "trust": trust_score,
        "runtime": runtime,
        "strategy": result["intermediate"]["strategy_used"],
        "retry_count": result["intermediate"]["retry_count"],
    })

benchmark_df = pd.DataFrame(results)

benchmark_df.head()


NameError: name 'qa_data' is not defined

## 10. Aggregate Statistics


In [20]:
benchmark_df.groupby("decision")[["trust", "runtime"]].mean()


NameError: name 'benchmark_df' is not defined

In [21]:
benchmark_df["decision"].value_counts()


NameError: name 'benchmark_df' is not defined

## 11. Failure Analysis

This section examines low-confidence cases.

In [22]:
low_conf = benchmark_df[
    benchmark_df["trust"] < 0.4
]

low_conf

NameError: name 'benchmark_df' is not defined

## 12. Final Discussion

This notebook implemented a reliable and adaptive agentic RAG framework aligned with the official Step 3 requirements.

Implemented capabilities:

- evidence sufficiency estimation
- groundedness verification
- contradiction detection
- clarification handling
- abstention
- self-reflection
- adaptive recovery
- trust estimation

The system dynamically adapts its behavior when failures or uncertainty are detected.

Limitations:

- heuristic-based reliability estimation
- lightweight contradiction detection
- simple answer generation

Future improvements:

- stronger verifier models
- semantic contradiction detection
- reinforcement learning orchestration
- retrieval strategy optimization
- claim-level grounding verification